# Experimento con el LLM: ¿respeta las reglas de producción?

**Responsable:** Persona 5

Se envían al LLM las reglas de nuestra gramática (`spanish/grammar.py`, sección 1.2 de
`docs/gramatica_es.md`) y el léxico, y se le pide responder **solo** con el JSON de la sección 2.6.

| Versión | Prompt | Salida |
| --- | --- | --- |
| `llm_zero` | solo reglas | `results/llm_zero.jsonl` |
| `llm_few` | reglas + ejemplos resueltos | `results/llm_few.jsonl` |

Todos los prompts y respuestas crudas quedan en `results/logs/llm.jsonl` (entregable obligatorio). Si
un `id` y `sistema` ya están en el log, se reutiliza la respuesta sin volver a llamar a la API.

**Claves de API:** nunca en el notebook. Van en un archivo `.env` en la raíz del repositorio (ya está
en `.gitignore`). Con `PROVIDER = "manual"` no hace falta clave: el notebook muestra el prompt y se
pega la respuesta de cualquier chat.

Preguntas a responder (van al informe):

1. ¿El LLM respeta las reglas o se desvía hacia su conocimiento del español?
2. ¿Dónde acierta y dónde falla?
3. ¿El few-shot mejora la precisión?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.formats import Prediction, load_test_set, save_predictions, results_path, load_predictions

items = load_test_set()
print(f"{len(items)} oraciones en el conjunto de prueba")

In [ ]:
# Configuración del experimento. Anotar en el informe el modelo exacto y la fecha.
PROVIDER = "manual"   # "manual" | "anthropic" | "openai" | "gemini"
MODEL = "manual"      # TODO(P5): nombre EXACTO del modelo
TEMPERATURE = 0

In [ ]:
from experiments import prompts

print(prompts.zero_shot(items[0].sentence))

In [ ]:
from experiments.llm_client import ask_llm
from spanish.tree import Node


def run(system: str, build_prompt) -> list[Prediction]:
    predictions = []
    for item in items:
        start = time.perf_counter()
        raw = ask_llm(build_prompt(item.sentence), id=item.id, system=system,
                      provider=PROVIDER, model=MODEL, temperature=TEMPERATURE)
        elapsed = (time.perf_counter() - start) * 1000
        answer = prompts.parse_answer(raw)
        if answer is None:
            predictions.append(Prediction(item.id, system, accepts=None, time_ms=elapsed,
                                          run_error="respuesta_no_json"))
            continue
        tree = answer["arbol"]
        predictions.append(Prediction(
            item.id, system,
            accepts=answer["acepta"],
            error_type=answer["tipo_error"] if answer["acepta"] is False else None,
            tags=answer["etiquetas"],
            subject=answer["sujeto"], verb=answer["verbo"], object=answer["objeto"],
            tree=tree, num_trees=1 if tree else None,
            time_ms=elapsed,
        ))
    return predictions

## 1. Zero-shot (solo reglas)

In [ ]:
llm_zero = run("llm_zero", prompts.zero_shot)
print("Guardado en", save_predictions("llm_zero", llm_zero, items))

## 2. Few-shot (reglas y ejemplos)

Los ejemplos están fijos en `experiments/prompts.py` (`EXAMPLES`) y no son oraciones del conjunto de prueba.

In [ ]:
print(prompts.few_shot(items[0].sentence))

In [ ]:
llm_few = run("llm_few", prompts.few_shot)
print("Guardado en", save_predictions("llm_few", llm_few, items))

## 3. Análisis

Las métricas completas las calcula la Persona 4 (`python -m evaluation.metrics`). Aquí van las
preguntas propias del experimento.

In [ ]:
from spanish.grammar import tree_shape_errors

by_id = {i.id: i for i in items}

for system in ("llm_zero", "llm_few"):
    if not results_path(system).exists():
        continue
    preds = load_predictions(system)
    no_json = [p.id for p in preds if p.run_error == "respuesta_no_json"]
    decided = [p for p in preds if p.accepts is not None]
    follows_rules = sum(p.accepts == by_id[p.id].in_grammar for p in decided)
    follows_spanish = sum(p.accepts == by_id[p.id].valid_spanish for p in decided)
    # Desviación: acepta español correcto que la gramática NO genera.
    drift = [p.id for p in decided if p.accepts and not by_id[p.id].in_grammar and by_id[p.id].valid_spanish]
    bad_trees = []
    for p in decided:
        if p.tree:
            try:
                if tree_shape_errors(Node.from_bracketed(p.tree)):
                    bad_trees.append(p.id)
            except ValueError:
                bad_trees.append(p.id)
    print(f"{system}: sin JSON {len(no_json)}, coincide con en_gramatica {follows_rules}/{len(decided)}, "
          f"con espanol_valido {follows_spanish}/{len(decided)}")
    print(f"  se desvía hacia el español en: {drift or 'ninguna'}")
    print(f"  árboles que no respetan la gramática: {bad_trees or 'ninguno'}")

### Conclusiones (para el informe)

TODO(P5):

1. ¿Respeta las reglas o se desvía? Comparar "coincide con en_gramatica" frente a "con espanol_valido" y citar ejemplos del log.
2. ¿Dónde acierta y dónde falla? Por nivel y por fenómeno (`fenomeno` del conjunto de prueba).
3. ¿El few-shot mejora la precisión? Comparar las dos filas.
4. Para la comparación de enfoques: robustez, ambigüedad, escalabilidad y costo (`tiempo_ms`, tokens y dinero).